In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import json
from uuid import uuid4
from pathlib import Path

import pandas as pd

from rich import print
from openai import OpenAI
from beir.datasets.data_loader import GenericDataLoader

/home/fahmi/freelance/project-2023-amsearch/.venv/lib/python3.11/site-packages/beir/datasets/data_loader.py:2: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
df_corpus = pd.read_json(f"../data/corpus.jsonl", lines=True)
df_corpus.head()

,id,title,content,published_at,word_count,source_url
0,1bcfc529-9788-4f8b-a8e4-c2780922cb9e,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...,2009-11-16,231,http://majalah-balebat.blogspot.com/2009/11/wa...
1,7ad64ffc-449a-4218-b3fb-a9ad92925068,Warga Bogor Mapag Taun anyar Islam,bogor - datang ton anyar islam 1431-hijréh pap...,2009-12-19,265,http://majalah-balebat.blogspot.com/2009/12/wa...
2,6c714fec-c629-4604-bfac-0096e1a0f624,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...,2011-06-12,1255,http://majalah-balebat.blogspot.com/2011/06/wa...
3,5669e653-63ab-49b4-be39-7bf07f2eefe7,Manfaat Olahraga Pikeun Kasehatan,anu ku urang tos terang olahraga teh penting p...,2018-11-28,246,http://tipscaras.blogspot.com/2017/02/contoh-a...
4,bfc35e44-876f-4ddd-a7c3-dbe6339a27f1,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g...",2023-07-24,69,https://basasunda.com/puisi-bahasa-sunda


## Open AI Batch Job

In [4]:
client = OpenAI()

In [5]:
def download_job(id: str, file_path: str):
    batch = client.batches.retrieve(id)
    print(batch)

    if (batch.status == "completed"):
        file_content = client.files.content(batch.output_file_id).content
        with open(file_path, "wb") as f:
            f.write(file_content)

## Generate Triplet

In [6]:
triplet_job_path = Path("../data/llm-gen/triplet/triplet_batch_result.jsonl")
download_job("batch_681df8a6dd208190b5f10ed38d14f958", triplet_job_path)

Batch(
    id='batch_681df8a6dd208190b5f10ed38d14f958',
    completion_window='24h',
    created_at=1746794662,
    endpoint='/v1/chat/completions',
    input_file_id='file-3ZSqqULYrT9kLkWXHHvTpS',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746796062,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746881062,
    failed_at=None,
    finalizing_at=1746795882,
    in_progress_at=1746794726,
    metadata=None,
    output_file_id='file-KfkU5ahA3tZnxBs9uNP6tV',
    request_counts=BatchRequestCounts(completed=1499, failed=0, total=1499)
)

In [7]:
triplet_total_tokens = 0

with (
    open(triplet_job_path, "r") as batch_file,
    open(f"../data/cleaned/triplet.jsonl", "w") as triplet_file,
):
    for line in batch_file:
        parsed = json.loads(line)
        model = json.loads(parsed["response"]["body"]["choices"][0]["message"]["content"])
        triplet_total_tokens += parsed["response"]["body"]["usage"]["total_tokens"]

        for item in model["triplets"]:
            data = {
                "query": item["query"],
                "positive": item.get("positive_passage") or item.get("relevant_passage"),
                "negative": item.get("negative_passage") or item.get("irrelevant_passage"),
            }

            json.dump(data, triplet_file)
            triplet_file.write("\n")

In [8]:
print(f"Total tokens: {triplet_total_tokens}")

Total tokens: 1577793

## Generate BEIR Dataset

In [9]:
beir_job_path = Path("../data/llm-gen/beir/beir_batch_result.jsonl")
download_job("batch_681e95db8c2c819094cc4302789affb5", beir_job_path)

Batch(
    id='batch_681e95db8c2c819094cc4302789affb5',
    completion_window='24h',
    created_at=1746834907,
    endpoint='/v1/chat/completions',
    input_file_id='file-K2PJRvTzH7VKLik1nABKuv',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746835727,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746921307,
    failed_at=None,
    finalizing_at=1746835621,
    in_progress_at=1746834970,
    metadata=None,
    output_file_id='file-XEpTH19ZCNSPNCzQjj5dMt',
    request_counts=BatchRequestCounts(completed=1499, failed=0, total=1499)
)

In [10]:
# load BEIR data mapping
beir_corpus_map = {}
with open(f"../data/llm-gen/beir/beir_map.jsonl", "r") as map_file:
    for line in map_file:
        parsed = json.loads(line)
        beir_corpus_map[parsed["custom_id"]] = parsed["doc_id"]

In [11]:
# generate BEIR corpus file
with open(f"../data/cleaned/corpus.jsonl", "w") as f:
    for row in df_corpus.itertuples():
        data = {
            "_id": row.id,
            "title": row.title,
            "text": row.content,
        }

        json.dump(data, f)
        f.write("\n")

In [14]:
beir_total_tokens = 0

with (
    open(beir_job_path, "r") as batch_file,
    open(f"../data/cleaned/queries.jsonl", "w") as queries_file,
    open(f"../data/cleaned/qrels.tsv", "w") as qrels_file,
):
    # write BEIR qrels header
    qrels_file.write("query-id\tcorpus-id\tscore\n")

    # process each batch
    for line in batch_file:
        try:
            parsed = json.loads(line)
            custom_id = parsed["custom_id"]

            beir = json.loads(parsed["response"]["body"]["choices"][0]["message"]["content"])
            beir_total_tokens += parsed["response"]["body"]["usage"]["total_tokens"]

            doc_id = beir_corpus_map[custom_id]
            for item in beir["queries"]:
                query_id = str(uuid4())

                qrels_file.write(f"{query_id}\t{doc_id}\t1\n")

                json.dump({"_id": query_id, "text": item["search_term"]}, queries_file)
                queries_file.write("\n")
        except:
            print(f"FAILED: {custom_id}")

FAILED: c11befe4-6922-4561-98b5-2f86e33fce12

In [15]:
print(f"Total tokens: {beir_total_tokens}")

Total tokens: 1281542

In [16]:
# validate BEIR dataset
corpus, queries, qrels = GenericDataLoader(
    data_folder="../data/cleaned", 
    qrels_file=f"../data/cleaned/qrels.tsv"
).load_custom()

100%|██████████| 1499/1499 [00:00<00:00, 147640.29it/s]
